# DNN em PyTorch

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import sys
import pickle
sys.path.append('.')

from text_features import clean_text, TFIDFVectorizer, encode_labels, decode_labels, CLASS_NAMES
from metrics import accuracy, f1_macro

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [2]:
TRAIN_PATH = '../datasets/dataset_train.csv'
TEST_PATH  = '../datasets/dataset_test.csv'
VALIDATION_PATH = '../datasets/dataset_val.csv'

df_train = pd.read_csv(TRAIN_PATH, sep=';')
df_test  = pd.read_csv(TEST_PATH, sep=';')
df_val   = pd.read_csv(VALIDATION_PATH, sep=';')

print(f'treino: {len(df_train)} exemplos')
print(f'teste:  {len(df_test)}  exemplos')
print()
print('distribuição de treino:')
print(df_train['Label'].value_counts())

treino: 17500 exemplos
teste:  3750  exemplos

distribuição de treino:
Label
Anthropic    3500
Meta         3500
Human        3500
Openai       3500
Google       3500
Name: count, dtype: int64


## *pré-processamento*

In [3]:
# ── limpeza de texto ───────────────────────────────────────────────────────
train_texts  = [clean_text(t) for t in df_train['Text'].tolist()]
train_labels = df_train['Label'].str.lower().tolist()

test_texts   = [clean_text(t) for t in df_test['Text'].tolist()]
test_labels  = df_test['Label'].str.lower().tolist()

# ── labels ─────────────────────────────────────────────────────────────────
Y_train_idx = encode_labels(train_labels)
Y_test_idx  = encode_labels(test_labels)

# ── PyTorch Dataset ────────
MAX_WORDS = 10000
vec = TFIDFVectorizer(max_words=MAX_WORDS)

class TfidfDataset(Dataset): 
    def __init__(self, texts, labels, vectorizer, train=True):
        self.texts = texts
        self.labels = labels
        self.vectorizer = vectorizer
        
        if train: 
            X_sparse = self.vectorizer.fit_transform(texts)
        else:
            X_sparse = self.vectorizer.transform(texts)
            
        X_dense = X_sparse.toarray() if not isinstance(X_sparse, np.ndarray) else X_sparse
        self.X = torch.tensor(X_dense, dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = self.X[idx]
        # Alterado para torch.long para suportar a CrossEntropyLoss (multi-classe)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

full_train_dataset = TfidfDataset(train_texts, Y_train_idx, vectorizer=vec, train=True)
test_dataset = TfidfDataset(test_texts, Y_test_idx, vectorizer=vec, train=False)

print(f'X_train: {full_train_dataset.X.shape}  Y_train: {len(Y_train_idx)}')
print(f'X_test:  {test_dataset.X.shape}   Y_test:  {len(Y_test_idx)}')

# ── PyTorch DataLoaders ────────────────────────────────────────────────────
train_size = int(0.85 * len(full_train_dataset))
val_size   = len(full_train_dataset) - train_size

train_ds, val_ds = random_split(full_train_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

X_train: torch.Size([17500, 10000])  Y_train: 17500
X_test:  torch.Size([3750, 10000])   Y_test:  3750


## *treino do modelo*

In [4]:
class FFNN(nn.Module):
    def __init__(self, input_dim, n_classes=5, topology=[256,128], dropout=0.3):
        super().__init__()
        layers = []
        if not topology:
            layers.append(nn.Linear(input_dim, n_classes))
        else:
            layers.append(nn.Linear(input_dim, topology[0]))
            layers.append(nn.ReLU())
            if dropout > 0: layers.append(nn.Dropout(dropout))
            
            for i in range(1, len(topology)):
                layers.append(nn.Linear(topology[i-1], topology[i]))
                layers.append(nn.ReLU())
                if dropout > 0: layers.append(nn.Dropout(dropout))
            
            layers.append(nn.Linear(topology[-1], n_classes))
            
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), correct / total

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_acc, patience_counter = 0, 0
    best_state = None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        
        if epoch % 10 == 0:
            print(f'[Epoch {epoch:03d}] train_acc: {tr_acc:.4f} | val_acc: {vl_acc:.4f}')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model

model_dnn = FFNN(input_dim=full_train_dataset.X.shape[1], n_classes=5, topology=[256, 128], dropout=0.3).to(device)
model_dnn = train_model(model_dnn, train_loader, val_loader, epochs=100, lr=0.001, patience=10)

[Epoch 010] train_acc: 0.9994 | val_acc: 0.8564
Early stopping at epoch 11


## *evaluation*

In [5]:
def get_metrics(model, loader):
    model.eval()
    all_preds,all_true = [],[]
    with torch.no_grad():
        for x, y in loader:
            outputs = model(x.to(device))
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_true.extend(y.numpy())
            
    Y_onehot = np.zeros((len(all_true), 5))
    Y_onehot[np.arange(len(all_true)), all_true] = 1
    P_onehot = np.zeros((len(all_preds), 5))
    P_onehot[np.arange(len(all_preds)), all_preds] = 1
    
    return accuracy(Y_onehot, P_onehot), f1_macro(Y_onehot, P_onehot), all_preds

train_loader_full = DataLoader(full_train_dataset, batch_size=64)
train_acc, train_f1, _ = get_metrics(model_dnn, train_loader_full)
test_acc, test_f1, test_preds = get_metrics(model_dnn, test_loader)

print(f'[TRAIN] accuracy: {train_acc:.4f}  f1-macro: {train_f1:.4f}')
print(f'[TEST] accuracy: {test_acc:.4f}   f1-macro: {test_f1:.4f}')

[TRAIN] accuracy: 0.9352  f1-macro: 0.9352
[TEST] accuracy: 0.8608   f1-macro: 0.8606


In [6]:
torch.save(model_dnn.state_dict(), '../models/model_dnn.pt')

with open('../vectorizers/vectorizer_dnn.pkl', 'wb') as f:
    pickle.dump(vec, f)

print('modelo e vectorizer guardados.')

modelo e vectorizer guardados.


## *validation*

In [7]:
with open('../vectorizers/vectorizer_dnn.pkl', 'rb') as f:
    vec = pickle.load(f)

input_dim = len(vec.word_index)
model_dnn = FFNN(input_dim=input_dim, n_classes=5, topology=[256,128], dropout=0.3).to(device)
model_dnn.load_state_dict(torch.load('models/model_dnn.pt', map_location=device))

val_texts  = [clean_text(t) for t in df_val['Text'].tolist()]
val_labels = df_val['Label'].str.lower().tolist()
Y_val_idx = encode_labels(val_labels)

val_dataset = TfidfDataset(val_texts, Y_val_idx, vectorizer=vec, train=False)
val_loader_final = DataLoader(val_dataset, batch_size=64)

val_acc, val_f1, val_preds = get_metrics(model_dnn, val_loader_final)
print(f'accuracy: {val_acc:.4f}  f1-macro: {val_f1:.4f}')

preds_decoded = decode_labels(val_preds)
df_results = pd.DataFrame({'ID': df_val['ID'], 'Predicted': preds_decoded, 'True': df_val['Label']})
print(df_results.to_string(index=False))

accuracy: 0.8699  f1-macro: 0.8697
     ID Predicted      True
   VL-1 anthropic Anthropic
   VL-2    google    Google
   VL-3    openai    Openai
   VL-4    openai    Openai
   VL-5     human    Openai
   VL-6 anthropic Anthropic
   VL-7 anthropic Anthropic
   VL-8    google    Google
   VL-9     human     Human
  VL-10      meta      Meta
  VL-11    google    Google
  VL-12      meta      Meta
  VL-13    openai    Openai
  VL-14    openai    Openai
  VL-15     human     Human
  VL-16    openai    Openai
  VL-17    openai    Openai
  VL-18    openai    Openai
  VL-19    google    Google
  VL-20     human     Human
  VL-21    openai    Openai
  VL-22    google    Google
  VL-23      meta      Meta
  VL-24      meta    Google
  VL-25    openai    Openai
  VL-26    openai     Human
  VL-27    openai    Openai
  VL-28     human     Human
  VL-29     human     Human
  VL-30     human     Human
  VL-31     human     Human
  VL-32    google    Google
  VL-33    google    Google
  VL-34    op